In [1]:
!pip install -q transformers accelerate shap scikit-learn

In [2]:
# 1. Load the dataset
from google.colab import files
from datasets import load_dataset
import pandas as pd

ds = load_dataset("nanyy1025/covid_fake_news")

# stack three parts into one table
parts = []
for name in ["train", "validation", "test"]:
    parts.append(ds[name].to_pandas())
data = pd.concat(parts)

# keep two columns name text and label  with real = 0 and fake = 1
data = data.rename(columns={"tweet" : "text"})[["text", "label"]]
data["label"] = data["label"].map({"real" : 0, "fake" : 1})

# save as one csv
data.to_csv("covid.csv", index=False)
CSV_NAME = "covid.csv"
print("Using:", CSV_NAME, "| rows:", len(data))
print(data["label"].value_counts())

README.md:   0%|          | 0.00/495 [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/1.25M [00:00<?, ?B/s]

val.csv:   0%|          | 0.00/411k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/423k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6420 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2140 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2140 [00:00<?, ? examples/s]

Using: covid.csv | rows: 10700
label
0    5600
1    5100
Name: count, dtype: int64


In [3]:
# 2. Settings
TEXT_COL = "text"
LABEL_COL = "label"
LABEL_NAMEs = {0 : "Real", 1 : "Fake"}

MODEL_NAME = "distilbert-base-multilingual-cased"
MAX_LEN = 128
EPOCHS = 3

In [4]:
# 3. Text cleaning
%%writefile text_cleaning.py
"""Text cleaning for noisy, code-mixed social-media text (English, Hindi, Hinglish).

The same function is used in training (notebook) and in the app, so the model
always sees text cleaned in the same way.
"""
import re
import unicodedata

def clean_text(text):
  text = unicodedata.normalize("NFKC", str(text))                   # unify look-alike characters
  text = re.sub(r"^\s*RT\s+", "", text, flags=re.IGNORECASE)        # remove the RT retweet marker
  text = re.sub(r"https?://\S+|www\.\S+", " ", text)                # remove links
  text = re.sub(r"@\w+", " ", text)                                        # remove @mentions
  text = re.sub(r"#(\w+)", r"\1", text)                                    # "#FakeNews" -> "FakeNews"
  text = re.sub("[\U0001F000-\U0001FAFF\u2600-\u27BF\uFE0F]", " ", text)   # remove emojis
  text = re.sub(r"([^\W\d_])\1{2,}", r"\1\1", text)                        # "sooooo" -> "soo"
  text = re.sub(r"^[\s:,\-]+", "", text)                                   # leftover ": " at the start
  return re.sub(r"\s+", " ", text).strip()                                 # tidy up spaces

Writing text_cleaning.py


In [5]:
from text_cleaning import clean_text

In [6]:
# 4. Load and clean the data, then split
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv(CSV_NAME)
df = df[[TEXT_COL, LABEL_COL]].dropna().copy()
df["clean"] = df[TEXT_COL].astype(str).apply(clean_text)
df = df[df["clean"].str.len() > 0]

df["y"] = df[LABEL_COL]
id2label = {0 : "Real", 1 : "Fake"}
label2id = {"Real" : 0, "Fake" : 1}

# 80% train, 10% validation, 10% test (same class balance in each)
train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df["y"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["y"], random_state=42)
print(len(train_df), len(val_df), len(test_df))

8560 1070 1070


In [7]:
# 5. Baseline TF-IDF and Naive Bayes(compare against)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.naive_bayes import MultinomialNB

tfidf = TfidfVectorizer(token_pattern=r"[\w\u0900-\u097F]+", ngram_range=(1,2),  min_df=1, max_features=50000)
X_train = tfidf.fit_transform(train_df["clean"])
baseline = MultinomialNB().fit(X_train, train_df["y"])

nb_pred = baseline.predict(tfidf.transform(test_df["clean"]))
nb_acc = accuracy_score(test_df["y"], nb_pred)
nb_f1 = f1_score(test_df["y"], nb_pred, average="macro")
print(f"Baseline Accuracy: {nb_acc:.3f}, macro F1: {nb_f1:.3f}")



Baseline Accuracy: 0.916, macro F1: 0.915


In [8]:
# 6. Fine-tune the multilingual transformer
import numpy as np
import torch
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                            DataCollatorWithPadding, Trainer, TrainingArguments)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=len(id2label), id2label=id2label, label2id=label2id
    )


class TextDataset(torch.utils.data.Dataset):
    """Holds tokenized posts and their labels."""

    def __init__(self, texts, labels):
        self.encodings = tokenizer(list(texts), truncation=True, max_length=MAX_LEN)
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {key: values[i] for key, values in self.encodings.items()}
        item["labels"] = int(self.labels[i])
        return item


train_ds = TextDataset(train_df["clean"], train_df["y"])
val_ds = TextDataset(val_df["clean"], val_df["y"])
test_ds = TextDataset(test_df["clean"], test_df["y"])


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "f1": f1_score(labels, preds, average="macro")}

args = TrainingArguments(
        output_dir="checkpoints",
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,       # keep the epoch with the best validation F1
        metric_for_best_model="f1",
        save_total_limit=1,
        report_to="none",
        fp16=torch.cuda.is_available(),
    )

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
  )
trainer.train()


config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  542MB            

model.safetensors: downloading bytes:           |  0.00B            

... (3 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1605, training_loss=0.12582992616100847, metrics={'train_runtime': 164.3117, 'train_samples_per_second': 156.288, 'train_steps_per_second': 9.768, 'total_flos': 540765038163456.0, 'train_loss': 0.12582992616100847, 'epoch': 3.0})

In [12]:
# 7. Comparision on test set
from sklearn.metrics import classification_report

output = trainer.predict(test_ds)
tr_pred = output.predictions.argmax(axis=-1)
tr_acc = accuracy_score(test_df["y"], tr_pred)
tr_f1 = f1_score(test_df["y"], tr_pred, average="macro")

results = pd.DataFrame({
    "Model" : ["TF-IDF + Naive Bayes", MODEL_NAME],
    "Accuracy" : [round(nb_acc, 3), round(tr_acc, 3)],
    "Macro F1" : [round(nb_f1, 3), round(tr_f1, 3)],
})

print(results.to_string(index=False))
print()
print(classification_report(test_df["y"], tr_pred, target_names=[id2label[i] for i in range(len(id2label))]))

                             Model  Accuracy  Macro F1
              TF-IDF + Naive Bayes     0.916     0.915
distilbert-base-multilingual-cased     0.961     0.961

              precision    recall  f1-score   support

        Real       0.97      0.96      0.96       560
        Fake       0.95      0.96      0.96       510

    accuracy                           0.96      1070
   macro avg       0.96      0.96      0.96      1070
weighted avg       0.96      0.96      0.96      1070



In [13]:
# 8. Save the model

trainer.save_model("fake_news_model")
tokenizer.save_pretrained("fake_news_model")
print("Saved to the folder: fake_news_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved to the folder: fake_news_model


In [14]:
!zip -qr fake_news_model.zip fake_news_model
files.download("fake_news_model.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [15]:
# 9. Predicition with SHAP

import shap
from transformers import pipeline

clf = pipeline(
    "text-classification",
    model="fake_news_model",
    top_k=None,                 # return the score for every label
    truncation=True,
    max_length=MAX_LEN,
    device=0 if torch.cuda.is_available() else -1,
)
explainer = shap.Explainer(clf)

samples = test_df["clean"].head(3).tolist()
shap_values = explainer(samples, max_evals=300)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
PartitionExplainer explainer: 4it [00:11, 11.06s/it]               


In [16]:
shap.plots.text(shap_values[0])

In [17]:
print(test_df["y"].iloc[0], "->", "Real" if test_df["y"].iloc[0] == 0 else "Fake")

0 -> Real


In [18]:
wrong = test_df[tr_pred != test_df["y"].values]
print("Wrong predictions:", len(wrong), "of", len(test_df))

for text, label in zip(wrong["clean"].head(20), wrong["y"].head(20)):
    print("True label:", id2label[label], "|", text[:200])
    print()

Wrong predictions: 42 of 1070
True label: Real | Arterial and venous thrombosis issues with coronavirus and the role of anticoagulation.

True label: Real | The gallery like many arts institutions across the country has been hit hard by the COVID-19 pandemic and is reportedly considering lay-offs thanks to a 75% drop in revenue

True label: Real | 2) Live infectious virus was NOT found. The authors’ finding does not indicate SARS-CoV-2 was transmitted from these surfaces. study shows infectious SARS-CoV-2 virus can survive for only up to 72 hrs

True label: Real | New Zealand is now at Alert Level 2. Our team of 5 million has united to beat COVID-19. Our success is not down to luck; it’s due to hard work from you all. TeamOfFiveMillion TīmaRimaMiriona

True label: Real | We had to ration our supplies but it wasn't good for us — it's like breathing through a dirty CPAP mask. ppe covid19

True label: Fake | A lengthy post purportedly contains advice on how to prevent infection from the n